# Face Recognition using CNN — Labeled Faces in the Wild (LFW)

**Name:** Palak Narang
**Reg No:** 23BCE11819

**Objective:** Build a CNN model to recognize faces from the LFW dataset with a target accuracy of 90%.

**Dataset:** LFW (Labeled Faces in the Wild) — 1,288 grayscale images of 7 public figures, 50×37 pixels

In [ ]:
!pip install tensorflow matplotlib numpy scipy scikit-learn seaborn

: 

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np

print("TensorFlow version:", tf.__version__)

In [ ]:

lfw = fetch_lfw_people(min_faces_per_person=70, resize=0.4)

print("Dataset shape:", lfw.images.shape)
print("Number of classes (people):", lfw.target_names.shape[0])
print("People in dataset:", lfw.target_names)

In [ ]:

print("Number of images per person:")
for i, name in enumerate(lfw.target_names):
    count = np.sum(lfw.target == i)
    print(f"  {name}: {count} images")

# visualizing some sample faces
plt.figure(figsize=(12, 8))
for i in range(14):
    plt.subplot(2, 7, i+1)
    plt.imshow(lfw.images[i * 90], cmap='gray')
    plt.title(lfw.target_names[lfw.target[i * 90]].split()[-1], fontsize=9)
    plt.xticks([])
    plt.yticks([])
plt.suptitle("Sample Faces from LFW Dataset", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# preprocessing the data
x = lfw.images
y = lfw.target

# sklearn already gives pixel values between 0 and 1, so no need to divide by 255
print("Pixel range - Min:", x.min(), "Max:", x.max())

# adding channel dimension for CNN (grayscale = 1 channel)
x = x.reshape(-1, 50, 37, 1)

# splitting into train and test (80-20 split)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

print("Training data shape:", x_train.shape)
print("Test data shape:", x_test.shape)

In [ ]:
# building the CNN model for face recognition
model = models.Sequential([
    # first conv block - 32 filters
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(50, 37, 1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # second conv block - 64 filters
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # third conv block - 128 filters
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # dense layers
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    # output layer - 7 people so 7 neurons with softmax
    layers.Dense(7, activation='softmax')
])

model.summary()

In [ ]:
# data augmentation - important since we have limited training data
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)
datagen.fit(x_train)

# compiling with adam optimizer
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# callbacks - reduce lr when accuracy plateaus and early stopping
lr_reducer = ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=8, min_lr=1e-6, verbose=1, mode='max')
early_stop = EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True, verbose=1, mode='max')

# training with data augmentation
history = model.fit(datagen.flow(x_train, y_train, batch_size=32),
                    epochs=80,
                    validation_data=(x_test, y_test),
                    callbacks=[lr_reducer, early_stop])

In [ ]:
# plotting accuracy and loss curves to check for overfitting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Training Accuracy')
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax1.set_title('Training vs Validation Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'], label='Training Loss')
ax2.plot(history.history['val_loss'], label='Validation Loss')
ax2.set_title('Training vs Validation Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print("\nFinal Training Accuracy: {:.2f}%".format(history.history['accuracy'][-1] * 100))
print("Final Validation Accuracy: {:.2f}%".format(history.history['val_accuracy'][-1] * 100))

if history.history['val_accuracy'][-1] >= history.history['accuracy'][-1]:
    print("\n No overfitting detected - validation accuracy is >= training accuracy")
else:
    gap = history.history['accuracy'][-1] - history.history['val_accuracy'][-1]
    if gap < 0.05:
        print("\n No significant overfitting - gap is only {:.2f}%".format(gap * 100))
    else:
        print("\n Possible overfitting - gap is {:.2f}%".format(gap * 100))

In [ ]:

test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=1)
print("\nTest Accuracy: {:.2f}%".format(test_accuracy * 100))
print("Test Loss: {:.4f}".format(test_loss))

# visualizing some predictions
predictions = model.predict(x_test[:15])

plt.figure(figsize=(15, 9))
for i in range(15):
    plt.subplot(3, 5, i+1)
    plt.imshow(x_test[i].reshape(50, 37), cmap='gray')
    predicted = lfw.target_names[np.argmax(predictions[i])].split()[-1]
    actual = lfw.target_names[y_test[i]].split()[-1]
    color = 'green' if predicted == actual else 'red'
    plt.xlabel(f"Pred: {predicted}\nActual: {actual}", color=color, fontsize=8)
    plt.xticks([])
    plt.yticks([])
plt.suptitle("Face Recognition Predictions (Green = Correct, Red = Wrong)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# classification report and confusion matrix
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_pred = np.argmax(model.predict(x_test), axis=1)
short_names = [name.split()[-1] for name in lfw.target_names]

print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=short_names))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=short_names, yticklabels=short_names)
plt.title('Confusion Matrix - LFW Face Recognition')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n Final Test Accuracy: {:.2f}% (Target was 90%)".format(test_accuracy * 100))